# 02: Ozaki Schemes

High-precision matmul from low-precision mul blocks.

- **Ozaki I:** split matrices into bounded-bit slices, find all cross-product matmuls, acc in FP64
- **Ozaki II:** (TODO) improved slicing + acc

In [1]:
import numpy as np
import math
from ozaki1 import (
    extract_slice,
    ozaki_split,
    ozaki1_matmul,
    compensated_matmul,
    row_max_abs,
)

## Ozaki

Scalar Veltkamp splits one float into two $\sim 26$-bit halves. Ozaki extends to matrix scale:

1. For each row, find max abs value, `mu`
2. Pick `sigma = 2^(ceil(log2(mu)) + 27)` (power of two, $\sim 2^27$ bigger than biggest element)
3. `slice = (sigma + A) - sigma` rounds every element to $\sim 26$ significant bits
4. `remainder = A - slice` holds what's left
5. Repeat on remainder

After k rounds: `A ≈ S1 + S2 + ... + Sk`, each slice has bounded-bit elements.

## extract_slice

Single row, one extraction round, sigma forces rounding.

In [2]:
row = np.array([[3.14159, 0.00272, 100.0, -42.13]])
mu = row_max_abs(row)

print(f"row: {row[0]}")
print(f"mu:  {mu[0]}")
print()

S, R = extract_slice(row, mu)

print(f"{'element':>10s} {'slice':>12s} {'remainder':>12s}")
for i in range(row.shape[1]):
    print(f"{row[0,i]:>10.5f} {S[0,i]:>12.5f} {R[0,i]:>12.2e}")

print()
print("small elements (0.00272) coarsely rounded relative to mu")
print("to be captured better in the next slice")

row: [ 3.14159e+00  2.72000e-03  1.00000e+02 -4.21300e+01]
mu:  100.0

   element        slice    remainder
   3.14159      3.14159    -1.18e-07
   0.00272      0.00272     1.21e-07
 100.00000    100.00000     0.00e+00
 -42.13000    -42.13000    -8.39e-07

small elements (0.00272) coarsely rounded relative to mu
to be captured better in the next slice


## Slicing

Remainder shrinks with each slice. More slices = more precision captured.

In [3]:
A = np.array([[3.14159265, 0.00271828, 100.0, -42.1337]])
n_slices = 4

slices = ozaki_split(A, n_slices)

print(f"{'':>10s}", end="")
for j in range(A.shape[1]):
    print(f" {'col '+str(j):>12s}", end="")

print()
for i, s in enumerate(slices):
    label = f"slice {i}" if i < n_slices - 1 else "remainder"
    print(f"{label:>10s}", end="")
    for j in range(A.shape[1]):
        print(f" {s[0,j]:>12.4e}", end="")

    print()

print()
reconstruction = sum(slices)
print(f"sum of slices == A: {np.allclose(reconstruction, A)}")

                  col 0        col 1        col 2        col 3
   slice 0   3.1416e+00   2.7199e-03   1.0000e+02  -4.2134e+01
   slice 1  -1.2831e-06  -1.5992e-06   0.0000e+00  -5.8289e-07
   slice 2  -6.6613e-15  -1.0509e-14   0.0000e+00   7.1054e-15
 remainder   0.0000e+00   0.0000e+00   0.0000e+00   0.0000e+00

sum of slices == A: True


## Ozaki I matmul

`A @ B = sum of all A_slice[i] @ B_slice[j]`

Each cross-product matmul has bounded-bit operands, so individual muls are more accurate. The acc in FP64 gives a better result than naive `A @ B`.

In [4]:
rng = np.random.default_rng(42)

sizes = [8, 32]
n_slices_list = [2, 3, 4, 6]

for n in sizes:
    A = rng.standard_normal((n, n))
    B = rng.standard_normal((n, n))

    C_naive = A @ B
    C_ref = compensated_matmul(A, B)

    err_naive = np.max(np.abs(C_naive - C_ref))
    print(f"n={n}:")
    print(f" {'naive fp64':<20s} max|err| = {err_naive:.3e}")

    for ns in n_slices_list:
        C_oz = ozaki1_matmul(A, B, n_slices=ns)
        err = np.max(np.abs(C_oz - C_ref))
        print(f" {'ozaki1 k='+str(ns):<20s} max|err| = {err:.3e}  ({ns*ns} matmuls)")

    print()

n=8:
 naive fp64           max|err| = 4.441e-16
 ozaki1 k=2           max|err| = 8.882e-16  (4 matmuls)
 ozaki1 k=3           max|err| = 4.441e-16  (9 matmuls)
 ozaki1 k=4           max|err| = 4.441e-16  (16 matmuls)
 ozaki1 k=6           max|err| = 4.441e-16  (36 matmuls)

n=32:
 naive fp64           max|err| = 6.217e-15
 ozaki1 k=2           max|err| = 3.553e-15  (4 matmuls)
 ozaki1 k=3           max|err| = 3.553e-15  (9 matmuls)
 ozaki1 k=4           max|err| = 3.553e-15  (16 matmuls)
 ozaki1 k=6           max|err| = 3.553e-15  (36 matmuls)



## The tradeoff

| slices (k) | matmuls | accuracy |
|---|---|---|
| 1 | 1 | same as naive |
| 2 | 4 | $\sim 1$ digit better |
| 4 | 16 | $\sim 2-3$ digits better |
| 6 | 36 | diminishing returns |

More slices = more compute, better precision. If each slice has $\sim 26$-bit elements, those fit in small fixed-point multipliers. You trade one expensive FP64 matmul for many cheap low-precision ones.

## Ozaki II

TODO.